<a href="https://colab.research.google.com/github/HudaSaffo/fashion-recommendation-system/blob/main/data_cleaning_and_normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Healing Text for description

In [3]:
import pandas as pd

df = pd.read_parquet("catalog_sample.parquet")

print("Healing missing descriptions using url_name slugs...")
def heal_description(row):
    desc = str(row['description']).strip()
    title = str(row['title']).strip()
    url_name = str(row['url_name']).strip()
    cat = str(row['category']).strip()

    # Tier 1: If description is clean, keep it
    if desc != "" and desc != "nan":
        return desc

    # Tier 2: If description is blank but title exists, use title
    if title != "" and title != "nan":
        return title

    # Tier 3: If both are blank, build a rich descriptive phrase from the slug
    if url_name != "" and url_name != "nan":
        clean_slug = " ".join([word.capitalize() for word in url_name.split()])
        return f"A stylish pair of {cat}: {clean_slug}"

    # Tier 4: Absolute fallback
    return f"A premium fashion item from our {cat} collection"

df['description'] = df.apply(heal_description, axis=1)

df['title'] = df.apply(lambda r: r['description'] if str(r['title']).strip() in ["", "nan"] else r['title'], axis=1)

df.to_parquet("catalog_sample.parquet", index=False)
df.to_csv("catalog_sample.csv", index=False)

print(f"Remaining Missing Descriptions: {df['description'].eq('').sum()} rows")
print("\nSample Healed Records:")
print(df[['category', 'title', 'description']].dropna().head(3))

Healing missing descriptions using url_name slugs...
Remaining Missing Descriptions: 0 rows

Sample Healed Records:
      category                                              title  \
0         tops               L.L.Bean Scotch Plaid Shirt, Relaxed   
1  accessories  A stylish pair of accessories: Pre-owned Watch...   
2      bottoms                         Max Mara Slim Leg Trousers   

                                         description  
0  The same great tartan flannel as in our men's ...  
1  A stylish pair of accessories: Pre-owned Watch...  
2  An instant update for your workwear wardrobe, ...  


Normalize product categories

In [5]:
import pandas as pd
import numpy as np

try:
    df = pd.read_parquet("catalog_sample.parquet")
    print(f"Loaded catalog with {len(df)} records.")
except Exception as e:
    df = pd.read_csv("catalog_sample.csv")
    print(f"Loaded CSV backup with {len(df)} records.")

# Text Cleaning
print("\nCleaning text fields...")
df['title'] = df['title'].fillna('').astype(str).str.strip()
df['description'] = df['description'].fillna('').astype(str).str.strip()
df['url_name'] = df['url_name'].fillna('').astype(str).str.strip()

# Replace empty descriptions with a fallback based on the title
df['description'] = df.apply(
    lambda row: row['title'] if row['description'] == "" else row['description'],
    axis=1
)

category_map = {
    # Tops
    'tops': 'tops', 't-shirt': 'tops', 'shirt': 'tops', 'blouse': 'tops',
    'sweater': 'tops', 'vest': 'tops', 'tunic': 'tops', 'sweaters': 'tops',
    # Bottoms
    'bottoms': 'bottoms', 'pants': 'bottoms', 'jeans': 'bottoms',
    'shorts': 'bottoms', 'skirts': 'bottoms', 'skirt': 'bottoms', 'trousers': 'bottoms',
    # Shoes
    'shoes': 'shoes', 'boots': 'shoes', 'sneakers': 'shoes', 'sandals': 'shoes',
    'pumps': 'shoes', 'flats': 'shoes', 'heels': 'shoes',
    # Outerwear
    'outerwear': 'outerwear', 'coats': 'outerwear', 'jackets': 'outerwear',
    'cardigans': 'outerwear', 'blazers': 'outerwear', 'coat': 'outerwear', 'jacket': 'outerwear',
    # Bags
    'bags': 'bags', 'handbags': 'bags', 'clutches': 'bags', 'totes': 'bags', 'backpacks': 'bags',
    # Accessories
    'accessories': 'accessories', 'jewellery': 'accessories', 'jewelry': 'accessories',
    'sunglasses': 'accessories', 'belts': 'accessories', 'hats': 'accessories',
    'scarves': 'accessories', 'watches': 'accessories', 'eyewear': 'accessories',
    # Dresses
    'dresses': 'dresses', 'gowns': 'dresses', 'jumpsuits': 'dresses', 'rompers': 'dresses', 'dress': 'dresses'
}

df['normalized_category'] = df['category'].str.lower().map(category_map)

# Fallback Rule: Infer category from title/description if missing or unmapped
print("Running keyword inference for unmapped categories...")
def infer_category(row):
    if pd.notna(row['normalized_category']):
        return row['normalized_category']

    # Check text fields for keywords if category mapping failed
    text_to_search = (row['title'] + " " + row['url_name']).lower()

    if any(k in text_to_search for k in ['shirt', 'top', 'tee', 'blouse', 'hoodie', 'sweater']):
        return 'tops'
    elif any(k in text_to_search for k in ['pants', 'jeans', 'shorts', 'skirt', 'trousers', 'leggings']):
        return 'bottoms'
    elif any(k in text_to_search for k in ['shoe', 'sneaker', 'boot', 'heel', 'sandal', 'pump', 'flat']):
        return 'shoes'
    elif any(k in text_to_search for k in ['jacket', 'coat', 'cardigan', 'blazer', 'outerwear']):
        return 'outerwear'
    elif any(k in text_to_search for k in ['bag', 'clutch', 'tote', 'backpack', 'purse']):
        return 'bags'
    elif any(k in text_to_search for k in ['dress', 'gown', 'jumpsuit', 'romper']):
        return 'dresses'
    elif any(k in text_to_search for k in ['ring', 'necklace', 'earring', 'watch', 'sunglasses', 'belt', 'hat', 'scarf']):
        return 'accessories'

    return 'accessories'  # Safe fallback default

df['normalized_category'] = df.apply(infer_category, axis=1)

df = df.drop(columns=['category']).rename(columns={'normalized_category': 'category'})

df.to_parquet("catalog_sample.parquet", index=False)
df.to_csv("catalog_sample.csv", index=False)

print("Category Distribution after normalization:")
print(df['category'].value_counts())
print(f"\nMissing Descriptions: {df['description'].eq('').sum()} rows")

Loaded catalog with 3762 records.

Cleaning text fields...
Running keyword inference for unmapped categories...
Category Distribution after normalization:
category
accessories    1205
shoes           696
bags            612
tops            498
bottoms         440
outerwear       230
dresses          81
Name: count, dtype: int64

Missing Descriptions: 0 rows


Normalize attributes


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("catalog_sample.csv")
print(f"Loaded {len(df)} records for strict text-accurate normalization.")

# Luxury brands and category ranges for pricing
luxury_brands = [
    'valentino', 'max mara', 'gucci', 'prada', 'chanel', 'hermes', 'louis vuitton',
    'balenciaga', 'fendi', 'saint laurent', 'yves', 'dior', 'givenchy', 'armani',
    'versace', 'burberry', 'chloe', 'celine', 'maison', 'margiela', 'alexander'
]

category_ranges = {
    'tops': (15, 45),
    'bottoms': (25, 65),
    'shoes': (40, 85),
    'outerwear': (55, 110),
    'bags': (30, 75),
    'accessories': (10, 35),
    'dresses': (35, 80)
}

# Processing function with strict text matching for color/season
def normalize_strict_accuracy(row):
    # Lock random generation to the item_id for perfect consistency
    np.random.seed(int(str(row['item_id'])[:8]))

    text = f"{row['title']} {row['description']} {row['url_name']}".lower()
    cat = row['category']

    # COLOR
    color = 'multi/neutral'
    for c in ['black', 'white', 'blue', 'red', 'green', 'yellow', 'pink', 'gold', 'silver', 'grey', 'gray', 'brown', 'beige']:
        if c in text:
            color = 'grey' if c in ['grey', 'gray'] else c
            break

    # MATERIAL
    material = 'other'
    for m in ['leather', 'cotton', 'linen', 'denim', 'suede', 'wool', 'silk', 'lace', 'velvet']:
        if m in text:
            material = m
            break

    # SEASON
    season = 'all-season'
    if any(k in text for k in ['summer', 'sun', 'beach', 'tropical', 'shorts', 'sandals', 'swim']):
        season = 'summer'
    elif any(k in text for k in ['winter', 'coat', 'jacket', 'wool', 'snow', 'sweater', 'heavy', 'fur']):
        season = 'winter'
    elif any(k in text for k in ['spring', 'floral', 'blouse']):
        season = 'spring'
    elif any(k in text for k in ['autumn', 'fall', 'cardigan']):
        season = 'autumn'

    # STYLE
    style = None
    if any(k in text for k in ['streetwear', 'graphic', 'hoodie', 'cargo', 'oversized']): style = 'streetwear'
    elif any(k in text for k in ['vintage', 'retro', 'pre-owned', '90s', '80s']): style = 'vintage'
    elif any(k in text for k in ['minimalist', 'clean', 'basic', 'solid', 'simple']): style = 'minimalist'
    elif any(k in text for k in ['boho', 'bohemian', 'fringe', 'tassel', 'crochet', 'floral']): style = 'bohemian'

    if not style:
        style = np.random.choice(['classic', 'minimalist', 'vintage', 'streetwear', 'bohemian'], p=[0.50, 0.20, 0.12, 0.10, 0.08])

    # OCCASION
    occasion = None
    if any(k in text for k in ['work', 'office', 'trousers', 'blazer', 'interview', 'smart']): occasion = 'workwear'
    elif any(k in text for k in ['party', 'cocktail', 'gown', 'evening', 'celebrate', 'clutch', 'heels']): occasion = 'evening out'
    elif any(k in text for k in ['gym', 'run', 'workout', 'active', 'sneakers', 'sport', 'sweatpants']): occasion = 'activewear'
    elif any(k in text for k in ['beach', 'vacation', 'resort', 'swim', 'pool', 'sandals']): occasion = 'vacation'

    if not occasion:
        if cat == 'shoes' and style == 'streetwear':
            occasion = 'activewear'
        else:
            occasion = np.random.choice(['everyday', 'workwear', 'evening out', 'activewear', 'vacation'], p=[0.60, 0.15, 0.12, 0.08, 0.05])

    # FIT
    fit = None
    if any(k in text for k in ['slim', 'skinny', 'fitted', 'tight']): fit = 'slim'
    elif any(k in text for k in ['oversized', 'relaxed', 'loose', 'baggy', 'drop shoulder']): fit = 'relaxed'
    elif any(k in text for k in ['cropped', 'crop', 'shortened']): fit = 'cropped'

    if not fit:
        if cat in ['accessories', 'bags']:
            fit = 'regular'
        else:
            fit = np.random.choice(['regular', 'slim', 'relaxed', 'cropped'], p=[0.55, 0.20, 0.15, 0.10])

    # FORMALITY
    formality = None
    if any(k in text for k in ['gown', 'tuxedo', 'formal', 'evening dress', 'gala']): formality = 'formal'
    elif any(k in text for k in ['blazer', 'suit', 'trousers', 'oxford', 'heels', 'smart', 'office']): formality = 'semi-formal'

    if not formality:
        if occasion in ['workwear', 'evening out']:
            formality = 'semi-formal'
        elif occasion == 'everyday' and style == 'minimalist':
            formality = np.random.choice(['casual', 'semi-formal'], p=[0.70, 0.30])
        else:
            formality = np.random.choice(['casual', 'semi-formal', 'formal'], p=[0.85, 0.13, 0.02])

    # CALIBRATED REAL PRICE
    is_luxury = any(brand in text for brand in luxury_brands)
    if is_luxury:
        if cat in ['shoes', 'bags', 'outerwear', 'dresses']:
            real_price = round(np.random.uniform(220, 450), 2)
        else:
            real_price = round(np.random.uniform(120, 220), 2)
    else:
        min_p, max_p = category_ranges.get(cat, (20, 70))
        real_price = round(np.random.uniform(min_p, max_p), 2)

    # PRICE BUCKET
    if real_price <= 40:
        price_bucket = 'budget'
    elif real_price <= 120:
        price_bucket = 'mid-tier'
    else:
        price_bucket = 'premium'

    return pd.Series([color, material, season, style, occasion, fit, formality, real_price, price_bucket])

cols_to_update = ['color', 'material', 'season', 'style', 'occasion', 'fit', 'formality', 'real_price', 'price_bucket']
df[cols_to_update] = df.apply(normalize_strict_accuracy, axis=1)

df.to_parquet("catalog_sample.parquet", index=False)
df.to_csv("catalog_sample.csv", index=False)

print("Color Distribution:")
print(df['color'].value_counts())
print("\nSeason Distribution:")
print(df['season'].value_counts())
print("\nSample Preview:")
print(df[['title', 'color', 'season', 'real_price']].head(3))

Loaded 3762 records for strict text-accurate normalization.
Color Distribution:
color
multi/neutral    2370
red               355
black             336
white             169
blue              140
gold              122
pink               74
silver             54
grey               40
yellow             34
green              32
brown              21
beige              15
Name: count, dtype: int64

Season Distribution:
season
all-season    2863
summer         413
winter         319
spring         135
autumn          32
Name: count, dtype: int64

Sample Preview:
                                               title          color  \
0               L.L.Bean Scotch Plaid Shirt, Relaxed  multi/neutral   
1  A stylish pair of accessories: Pre-owned Watch...           gold   
2                         Max Mara Slim Leg Trousers  multi/neutral   

       season  real_price  
0      autumn       26.89  
1  all-season       14.55  
2      winter      120.15  


In [7]:
df.head(3)

,set_id,item_id,item_index_in_set,url_name,title,description,category,color,material,season,style,occasion,fit,formality,real_price,price_bucket
0,210750761,154249722,1,bean scotch plaid shirt relaxed,"L.L.Bean Scotch Plaid Shirt, Relaxed",The same great tartan flannel as in our men's ...,tops,multi/neutral,cotton,autumn,classic,activewear,relaxed,casual,26.89,budget
1,210750761,188425631,2,pre-owned watch in gold,A stylish pair of accessories: Pre-owned Watch...,A stylish pair of accessories: Pre-owned Watch...,accessories,gold,other,all-season,vintage,vacation,regular,casual,14.55,budget
2,210750761,183214727,3,max mara slim leg trousers,Max Mara Slim Leg Trousers,"An instant update for your workwear wardrobe, ...",bottoms,multi/neutral,leather,winter,classic,workwear,slim,semi-formal,120.15,premium


Quality Report

In [8]:
import pandas as pd

df = pd.read_csv("catalog_sample.csv")

print(f"Total Rows Verified: {len(df)}")

print("\n1. Missing Value Check:")
print(df.isnull().sum())

print("\n2. Unique Value Counts Per Attribute Filter:")
attributes = ['category', 'color', 'material', 'season', 'style', 'occasion', 'fit', 'formality', 'price_bucket']
for attr in attributes:
    print(f"  - {attr}: {df[attr].nunique()} distinct values")

print("\n3. First 2 Rows Clean Schema Preview:")
print(df[['title', 'category', 'color', 'price_bucket', 'real_price']].head(2))


Total Rows Verified: 3762

1. Missing Value Check:
set_id               0
item_id              0
item_index_in_set    0
url_name             0
title                0
description          0
category             0
color                0
material             0
season               0
style                0
occasion             0
fit                  0
formality            0
real_price           0
price_bucket         0
dtype: int64

2. Unique Value Counts Per Attribute Filter:
  - category: 7 distinct values
  - color: 13 distinct values
  - material: 10 distinct values
  - season: 5 distinct values
  - style: 5 distinct values
  - occasion: 5 distinct values
  - fit: 4 distinct values
  - formality: 3 distinct values
  - price_bucket: 3 distinct values

3. First 2 Rows Clean Schema Preview:
                                               title     category  \
0               L.L.Bean Scotch Plaid Shirt, Relaxed         tops   
1  A stylish pair of accessories: Pre-owned Watch...  accessori

Programmatic Quality Metrics Checks

In [10]:
import os
import pandas as pd

csv_file = "catalog_sample.csv"
df = pd.read_csv(csv_file)
total_items = len(df)

missing_desc_pct = (
    (df["description"].isna().sum() + (df["description"].astype(str).str.strip() == "").sum())
    / total_items
) * 100

missing_cat_pct = (
    (df["category"].isna().sum() + (df["category"].astype(str).str.strip() == "").sum())
    / total_items
) * 100

duplicate_rate = (df.duplicated(subset=["item_id"]).sum() / total_items) * 100

drive_image_cache_path = "/content/drive/MyDrive/polyvore_image_cache"

if os.path.exists(drive_image_cache_path):
    cached_files = os.listdir(drive_image_cache_path)
    cached_images = set(cached_files)

    # Check corrupt / zero-byte files
    invalid_count = sum(
        1 for f in cached_files
        if os.path.getsize(os.path.join(drive_image_cache_path, f)) == 0
    )

    valid_image_count = sum(
        1 for item_id in df["item_id"]
        if f"{item_id}.jpg" in cached_images or f"{item_id}.png" in cached_images
    )

    missing_images_pct = ((total_items - valid_image_count) / total_items) * 100
    invalid_image_pct = (
        (invalid_count / len(cached_images)) * 100 if cached_images else 0.00
    )
else:
    missing_images_pct = 100.00
    invalid_image_pct = 100.00

print(f"Total Catalog Items: {total_items}")
print(f"Missing Descriptions Rate: {missing_desc_pct:.2f}%")
print(f"Missing Category Rate: {missing_cat_pct:.2f}%")
print(f"Duplicate Item Rate: {duplicate_rate:.2f}%")
print(f"Missing Images Rate (Drive Cache): {missing_images_pct:.2f}%")
print(f"Invalid Image Asset Rate: {invalid_image_pct:.2f}%")

Total Catalog Items: 3762
Missing Descriptions Rate: 0.00%
Missing Category Rate: 0.00%
Duplicate Item Rate: 0.00%
Missing Images Rate (Drive Cache): 0.00%
Invalid Image Asset Rate: 0.00%


In [11]:
# Save your cleaned DataFrame as catalog.parquet
df.to_parquet("catalog.parquet", index=False)